# AgriPrévision — planification et suivi de l’irrigation

Ce notebook appelle le même moteur que l’application et l’API. Les exemples sont **synthétiques** : ils vérifient des calculs et ne constituent pas un conseil validé pour une parcelle. Les résultats historiques v24 restent conservés dans leur protocole distinct.

Le plan sépare demande sans stress, consommation ajustée, apport net, volume brut au champ et prélèvement à la source. Il couvre le semis à la récolte et conserve le bilan après correction des observations.

## 1. Environnement et entrées

Prérequis : Python 3 et Node.js 24, dépôt complet. Aucun paquet Python externe n’est nécessaire. Les propriétés de sol, coefficients et météo du fichier d’exemple sont des hypothèses identifiées. Remplacer les entrées par un export de campagne pour examiner une autre situation.

In [1]:
import json, sys, copy
from pathlib import Path
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts/run-water-plan.mjs').exists())
sys.path.insert(0, str(root / 'science-source/seasonal'))
from irrigation import plan_irrigation
payload = json.loads((root / 'science-source/seasonal/example-campaign.json').read_text())
config = payload['config']
print('Pays :', config['country'], '| Culture :', config['crop'])
print('Surface (m²) :', config['areaM2'], '| Source sol :', config['soil']['source'])
print('Scénario météo :', config['fallback']['label'])

Pays : TGO | Culture : Maïs
Surface (m²) : 1000 | Source sol : Profil synthétique de démonstration, aucune analyse réelle
Scénario météo : Scénario sec hypothétique, sans pluie ; aucune prévision météo


## 2. Cycle et bilan de référence

Le moteur réalise un bilan quotidien avec croissance racinaire, pluie, irrigation, drainage et évapotranspiration. Les efficacités d’application et de transport portent sur des frontières distinctes. Le scénario sec continu n’est pas une prévision de pluie future.

In [2]:
plan = plan_irrigation(config)
assert plan['complete']
print('Version :', plan['engineVersion'], '| Fin du cycle :', plan['horizonEnd'])
print(json.dumps(plan['totals'], indent=2, ensure_ascii=False))
print('Jours calculés :', len(plan['days']))

Version : water-balance-1.1.0 | Fin du cycle : 2027-03-02
{
  "demandMm": 536.25,
  "actualEtMm": 536.25,
  "unmetEtMm": 0,
  "netMm": 529.5,
  "recordedSourceM3": 0,
  "remainingSourceM3": 706,
  "sourceM3": 706,
  "fieldM3": 706,
  "remainingBudgetM3": 194
}
Jours calculés : 125


In [3]:
max_residual = max(abs(d['balanceResidualMm']) for d in plan['days'])
assert max_residual < 1e-8
assert all(-1e-8 <= d['stockEndMm'] <= d['tawMm'] + 1e-8 for d in plan['days'])
for p in plan['weeks']:
    expected = p['stockStartMm'] + p['rootGainMm'] + p['assimilationMm'] + p['rainMm'] - p['runoffMm'] + p['netMm'] - p['drainageMm'] - p['actualEtMm']
    assert abs(p['stockEndMm'] - expected) < 1e-8
print('Écart maximal de conservation (mm) :', max_residual)
print('Contrôles de conservation et de stockage : réussis')

Écart maximal de conservation (mm) : 1.4210854715202004e-14
Contrôles de conservation et de stockage : réussis


## 3. Synthèse par stade

Les stocks ne s’additionnent pas d’une période à l’autre. Les volumes sont ici calculables parce que surface et efficacités sont renseignées explicitement. Les intervalles affichés sont issus de la trajectoire simulée.

In [4]:
for p in plan['stages']:
    print(p['label'], '| besoin :', round(p['demandMm'], 2), 'mm | apport net :', round(p['netMm'], 2), 'mm | source :', round(p['sourceM3'], 2), 'm³')
print('Intervalles proposés (jours) :', plan['intervalDays'])

Implantation | besoin : 30 mm | apport net : 26.37 mm | source : 35.16 m³
Croissance | besoin : 131.25 mm | apport net : 125.13 mm | source : 166.84 m³
Floraison et formation du grain | besoin : 240 mm | apport net : 252 mm | source : 336 m³
Maturation | besoin : 135.0 mm | apport net : 126 mm | source : 168 m³
Intervalles proposés (jours) : {'min': 3, 'max': 10}


## 4. Pluie enregistrée et correction tardive

L’événement suivant est une **mesure fictive pour le test**. Une observation prioritaire remplace la pluie météo de la même date. La correction conserve le même identifiant ; l’ancien plan Python est conservé séparément pour comparaison.

In [5]:
rain = dict(id='rain-fixture', date=config['sowingDate'], kind='rain', amount=10, unit='mm', point='field', status='measured', source='Mesure synthétique de test', recordedAt=config['asOfDate']+'T08:00:00Z')
after_rain = plan_irrigation(config, [rain])
corrected_rain = dict(rain, amount=25)
after_correction = plan_irrigation(config, [corrected_rain])
print('Volume source initial :', plan['totals']['sourceM3'])
print('Après pluie de test :', after_rain['totals']['sourceM3'])
print('Après correction :', after_correction['totals']['sourceM3'])
assert after_correction['days'][0]['rainMm'] == 25
assert all(abs(d['balanceResidualMm']) < 1e-8 for d in after_correction['days'])

Volume source initial : 706
Après pluie de test : 690.6666666666666
Après correction : 690


## 5. Irrigation réelle, intervention prévue et annulation

La quantité mesurée est traduite au point de mesure indiqué. Une irrigation seulement planifiée ou annulée ne modifie pas le réalisé. Ce cas vérifie le comportement du logiciel ; aucun apport de cet exemple ne constitue une prescription.

In [6]:
irrigation = dict(id='irrigation-fixture', date=config['sowingDate'], kind='irrigation', amount=8, unit='m3', point='field', status='declared', source='Compteur synthétique de test', recordedAt=config['asOfDate']+'T09:00:00Z')
with_irrigation = plan_irrigation(config, [irrigation])
planned_only = plan_irrigation(config, [dict(irrigation, status='planned')])
assert planned_only['totals'] == plan['totals']
assert with_irrigation['days'][0]['intervention']['type'] == 'recorded'
print('Volume brut au champ enregistré (m³) :', with_irrigation['days'][0]['intervention']['fieldM3'])
print('Apport net correspondant (mm) :', with_irrigation['days'][0]['irrigationNetMm'])

Volume brut au champ enregistré (m³) : 8
Apport net correspondant (mm) : 6


## 6. Eau limitée et données insuffisantes

Un quota nul demeure distinct d’une disponibilité inconnue. Si la météo ne permet plus de prolonger le bilan, aucun volume complet du cycle n’est annoncé. Une valeur manquante n’est jamais remplacée silencieusement par zéro.

In [7]:
limited = copy.deepcopy(config)
limited['system']['sourceBudgetM3'] = 0
limited_plan = plan_irrigation(limited)
assert limited_plan['totals']['sourceM3'] == 0
assert limited_plan['totals']['unmetEtMm'] > 0
print('Déficit de consommation sous quota nul (mm) :', round(limited_plan['totals']['unmetEtMm'], 2))
incomplete = copy.deepcopy(config)
incomplete['fallback'] = None
blocked_plan = plan_irrigation(incomplete)
assert not blocked_plan['complete']
assert blocked_plan['totals']['sourceM3'] is None
print('Données nécessaires :', blocked_plan['missing'])

Déficit de consommation sous quota nul (mm) : 455.3
Données nécessaires : ['Météo incomplète le 2026-10-29 : pluie et scénario de repli manquants. Le bilan ne peut pas être prolongé.']


## 7. Paramètres propres à chaque culture

Le catalogue conserve l’origine des coefficients, les contextes des durées et les valeurs manquantes. Les profils hydriques sont des scénarios, sans validation locale implicite. Les exemples dont le total publié diffère de la somme des phases exigent une saisie locale explicite.


In [8]:
from irrigation import crop_catalog, apply_crop, fertilization_reference
catalog = crop_catalog()
print(len(catalog), 'fiches culturales ; toutes ne disposent pas de paramètres hydriques.')
maize_config = apply_crop(copy.deepcopy(config), 'maize')
onion_config = apply_crop(copy.deepcopy(config), 'onion')
assert maize_config['stages'] != onion_config['stages']
print('Kc initial maïs / oignon :', maize_config['stages'][0]['kcStart'], '/', onion_config['stages'][0]['kcStart'])
assert apply_crop(config, 'yam')['stages'] == []
print('Igname : paramètres hydriques non acquis, aucun transfert du maïs.')


23 fiches culturales ; toutes ne disposent pas de paramètres hydriques.
Kc initial maïs / oignon : 0.3 / 0.7
Igname : paramètres hydriques non acquis, aucun transfert du maïs.


## 8. Chimie du sol et référence de fertilisation

Exemple fictif de parcelle de 1 000 m² en Kozah. La recette territoriale ITRA/FertiTogo (mars 2021, p.6) fournit 250 kg/ha de NPK 15-15-15 et 100 kg/ha d’urée, avec composition à vérifier. Cela ne constitue pas une dose adaptée à cette parcelle : la réponse sol–culture n’est pas calibrée. La mesure Olsen-Dabin ci-dessous est synthétique et ne modifie arbitrairement ni la dose ni le rendement. P assimilable et P₂O₅ des engrais restent distincts.


In [9]:
maize_config['territory'] = 'Kozah'
maize_config['areaM2'] = 1000
maize_config['soilAnalyses'] = [dict(source='Laboratoire fictif pour démonstration', date=maize_config['asOfDate'], availableOn=maize_config['asOfDate'], topCm=0, bottomCm=20, status='measured', measurements=[dict(parameter='availableP', value=8, unit='mg/kg', method='Olsen-Dabin')])]
context_note = 'Exemple arithmétique Kozah ; analyse fictive, adaptation au sol non établie.'
reference = fertilization_reference(maize_config, 'tgo-kozah-maize', context_note, [50, None])
assert reference['products'][0]['parcelKg'] == [25, 25]
assert reference['products'][0]['remainingParcelKg'] == [20, 20]
assert reference['products'][1]['remainingKgHa'] is None
assert reference['personalizedToSoil'] is False
maize_config['nutritionReference'] = dict(id='tgo-kozah-maize', contextNote=context_note, alreadyKgHa=[50, None])
assert plan_irrigation(maize_config)['nutrition'] == reference
print('Quantités pour la parcelle :', [p['parcelKg'] for p in reference['products']], 'kg')
print('N–P₂O₅–K₂O de la recette (kg/ha) :', reference['nutrientKgHa'])
print('Même calcul que l’interface et l’API ; aucune prescription personnalisée au sol.')


Quantités pour la parcelle : [[25, 25], [10, 10]] kg
N–P₂O₅–K₂O de la recette (kg/ha) : [[83.5, 83.5], [37.5, 37.5], [37.5, 37.5]]
Même calcul que l’interface et l’API ; aucune prescription personnalisée au sol.


## 9. Interprétation et limites

Ces vérifications portent sur les unités, la conservation et le rejeu. Elles ne valident pas le calendrier sur le terrain. Il reste à qualifier les données par zone, culture et système, à comparer des cas agronomiques externes et à suivre des parcelles en pilote. Le schéma générique exclut rizières submergées et nappes proches. Les pertes d’application sont agrégées ; seuls les excès supplémentaires au-delà du stockage sont drainés dans le bilan racinaire.

Sources : [FAO 56, bilan et stress](https://www.fao.org/4/X0490E/x0490e0e.htm), [coefficients culturaux](https://www.fao.org/4/X0490E/x0490e0b.htm), [efficacités](https://www.fao.org/4/t7202e/t7202e08.htm).

## Couverture auditée et capacités réelles
Les nombres suivants proviennent des archives vérifiées, toutes cultures retenues confondues. Ils ne sont ni des rendements de champs ni des performances saisonnières. Le même registre est appelé par l’application et l’API.


In [10]:
from irrigation import country_capabilities
for country in ["TGO", "BEN", "BFA", "GHA", "CIV", "SEN"]:
    capability = country_capabilities(country, "maize")
    history = capability["historicalData"]
    print(country, "observations territoriales =", history["observations"], "dernière année =", history["lastYear"], "prévision saisonnière =", capability["seasonalYieldForecast"]["status"])


TGO observations territoriales = 2014 dernière année = 2019 prévision saisonnière = blocked
BEN observations territoriales = 11412 dernière année = 2023 prévision saisonnière = blocked
BFA observations territoriales = 9313 dernière année = 2023 prévision saisonnière = blocked
GHA observations territoriales = 1025 dernière année = 2022 prévision saisonnière = blocked
CIV observations territoriales = None dernière année = None prévision saisonnière = blocked
SEN observations territoriales = 9317 dernière année = 2015 prévision saisonnière = blocked
